# 06 — Visualizations
Generate figures for the report: training curves, comparison bar charts, and gameplay snapshots.

In [ ]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.utils.plotting import plot_training_curves, plot_comparison
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

In [ ]:
# Load training logs from TensorBoard
def load_tb_rewards(log_dir):
    ea = EventAccumulator(log_dir)
    ea.Reload()
    tag = 'rollout/ep_rew_mean'
    if tag in ea.scalars.Keys():
        events = ea.scalars.Items(tag)
        return [e.value for e in events]
    return []

rewards = {
    'Pixel DQN': load_tb_rewards('../logs/pixel_dqn'),
    'Pixel PPO': load_tb_rewards('../logs/pixel_ppo'),
    'Symbolic DQN': load_tb_rewards('../logs/symbolic_dqn'),
    'Symbolic PPO': load_tb_rewards('../logs/symbolic_ppo'),
}

In [ ]:
# Training curves
fig = plot_training_curves(rewards, title='Training Curves — All Agents',
                           save_path='../results/training_curves.png')
plt.show()

In [ ]:
# Load evaluation results
import pandas as pd
df = pd.read_csv('../results/evaluation_summary.csv', index_col=0)
df

In [ ]:
# Comparison bar chart (manually rebuild results dict for plotting)
# This requires re-running evaluation or loading saved episode data
# For now, create a simple bar chart from the CSV
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Parse mean reward
mean_rewards = df['Mean Reward'].str.split(' ± ', expand=True).astype(float)
axes[0].bar(df.index, mean_rewards[0], yerr=mean_rewards[1], capsize=5,
            color=plt.cm.Set2.colors[:len(df)])
axes[0].set_ylabel('Mean Reward')
axes[0].set_title('Reward Comparison')
axes[0].tick_params(axis='x', rotation=20)

# Flag rate
flag_rates = df['Flag Rate'].str.rstrip('%').astype(float) / 100
axes[1].bar(df.index, flag_rates, color=plt.cm.Set2.colors[:len(df)])
axes[1].set_ylabel('Flag Rate')
axes[1].set_title('Level Completion Rate')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../results/comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()